In [1]:
import os
import random

# 1. CREATE MASTER INFRASTRUCTURE STORAGE
os.makedirs('gateway_telemetry', exist_ok=True)

# Define simulated inbound API microservice request payloads
payload_templates = [
    {"endpoint": "/api/v1/checkout", "format": "JSON", "size_kb": 12, "header": "X-Auth-Token: Valid"},
    {"endpoint": "/api/v1/user/profile", "format": "JSON", "size_kb": 4, "header": "X-Auth-Token: Valid"},
    {"endpoint": "/api/v1/admin/debug", "format": "MALFORMED_XML", "size_kb": 4500, "header": "X-Auth-Token: Missing"},
    {"endpoint": "/api/v1/payments", "format": "RAW_TEXT_INJECTION", "size_kb": 8200, "header": "X-Auth-Token: Expired"}
]

# 2. RUN CLOUD TRAFFIC ROUTER (3,000 continuous payload screening cycles)
routing_ledger = []
successful_routes = 0
quarantined_payloads = 0

for i in range(1, 3001):
    request_id = f"REQ-{i:05d}"

    # Asynchronously ingest a raw web request payload into the gateway
    request = random.choice(payload_templates)
    url_path = request["endpoint"]
    data_format = request["format"]
    payload_size = request["size_kb"]
    auth_header = request["header"]

    # Architecture Compliance Rule: Enforce backend structural constraints
    # Requests that are non-JSON, over 1000KB, or missing tokens violate core standards
    is_oversized = payload_size > 1000
    is_invalid_format = data_format != "JSON"
    is_unauthenticated = "Missing" in auth_header or "Expired" in auth_header

    # Core Gateway Switch Routing Logic
    if is_oversized or is_invalid_format or is_unauthenticated:
        routing_destination = "QUARANTINE_ISOLATION_SERVER"
        http_status_code = 400
        quarantined_payloads += 1
    else:
        routing_destination = "CORE_APPLICATION_CLUSTER"
        http_status_code = 200
        successful_routes += 1

    routing_ledger.append(
        f"{request_id} | Path: {url_path} | Format: {data_format} | Size: {payload_size}KB | "
        f"Route_To: {routing_destination} | HTTP_Status: {http_status_code}\n"
    )

# 3. SAVE THE IMMUTABLE GATEWAY METRICS
with open('gateway_telemetry/api_routing_audit.log', 'w') as f:
    f.writelines(routing_ledger)

print("--- CLOUD MICROSERVICES API GATEWAY INITIALIZED ---")
print(f"Total Inbound Production API Requests Screened: 3,000")
print(f"Successfully Routed to Core Microservices: {successful_routes}")
print(f"Quarantined & Isolated Malformed Requests: {quarantined_payloads}")


--- CLOUD MICROSERVICES API GATEWAY INITIALIZED ---
Total Inbound Production API Requests Screened: 3,000
Successfully Routed to Core Microservices: 1466
Quarantined & Isolated Malformed Requests: 1534


# Portfolio Project Phase 8: Cloud Software Architecture & API Gateway Engineering

## 📘 Code Explainer: API Gateway Payload Screening & Traffic Router
*This document breaks down the cloud backend load-balancing and routing logic line-by-line for non-technical stakeholders.*

### 1. Ingesting Asynchronous Cloud Traffic
* **`payload_templates = [...]`**: This maps out our simulated web socket interface, replicating raw endpoint transactions hitting an enterprise system—tracking sizes, payload string formatting (JSON vs. Malformed XML), and authentication header integrity.
* **`random.choice(payload_templates)`**: This mimics a highly transactional live cloud microservices application processing traffic spikes from around the globe.

### 2. Algorithmic Traffic Slicing (The Gateway Architecture)
* **`is_oversized = payload_size > 1000`**: This line serves as our infrastructure protection gate. It flags any payload over 1 Megabyte to prevent a Distributed Denial of Service (DDoS) attack or an intentional memory overflow payload from hitting internal microservices.
* **`if is_oversized or is_invalid_format or is_unauthenticated:`**: This is our structural switch routing matrix. If a request breaches any parameter, the gateway drops its standard destination and dynamically routes it to `QUARANTINE_ISOLATION_SERVER` with an HTTP 400 Bad Request code, preventing downstream service outages.

---

## 🤖 Autonomous AI Agent Architecture: The Intelligent Cloud API Traffic Router Agent
We propose an autonomous, multi-step cloud software engineering agent designed using an agentic framework (like LangGraph) to act as a real-time, automated Site Reliability Engineer (SRE).

